# Predictive Maintenance & Machine Failure Prediction

**Dataset:** AI4I 2020 Predictive Maintenance Dataset  
**Objective:** Predict machine failure from operating conditions and sensor readings.

This notebook documents the complete modelling workflow: data understanding, exploratory analysis, leakage-aware feature selection, model development, threshold selection, final evaluation, explainability, and model export.

> Run the notebook from top to bottom. The test set is reserved for final evaluation and should not be used to tune the model or decision threshold.

## 1. Imports and setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

sns.set_theme(style="whitegrid")

## 2. Load the dataset

In [ ]:
# Resolve project root whether the notebook is run from the project root
# or from the notebook/ directory.
BASE_DIR = Path.cwd()
if BASE_DIR.name == "notebook":
    BASE_DIR = BASE_DIR.parent

DATA_PATH = BASE_DIR / "data" / "pm.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

## 3. Understand the data

In [ ]:
df.describe()

In [ ]:
df.head()

In [ ]:
df.isna().sum()

In [ ]:
df.shape

In [ ]:
df.duplicated().sum()

In [ ]:
# Inspect target and failure-mode label distributions
failure_columns = [
    "Machine failure", "TWF", "HDF", "PWF", "OSF", "RNF"
]

for col in failure_columns:
    print(f"\\n{col}")
    print(df[col].value_counts(dropna=False))

## 4. Exploratory data analysis

In [ ]:
numerical_columns = [
    'Air temperature [K]',
    'Process temperature [K]',
    'Rotational speed [rpm]',
    'Torque [Nm]',
    'Tool wear [min]'
]

df[numerical_columns].hist(
    figsize=(14, 10),
    bins=30,
    edgecolor='black'
)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(14, 8))

for i, col in enumerate(numerical_columns, 1):
    plt.subplot(2, 3, i)
    sns.boxplot(y=df[col])
    plt.title(col)

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 7))

correlation = df[numerical_columns + ['Machine failure']].corr()

sns.heatmap(
    correlation,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)

plt.title('Sensor Correlation Heatmap')
plt.show()

In [ ]:
plt.figure(figsize=(15, 10))

for i, col in enumerate(numerical_columns, 1):

    plt.subplot(2, 3, i)

    sns.boxplot(
        data=df,
        x='Machine failure',
        y=col
    )

    plt.title(f'{col} vs Machine Failure')

plt.tight_layout()
plt.show()

In [ ]:
failure_analysis = df.groupby('Machine failure')[
    numerical_columns
].agg(['mean', 'median', 'std'])

print(failure_analysis.T)

In [ ]:
comparison = df.groupby('Machine failure')[
    numerical_columns
].mean().T

comparison.columns = ['Normal', 'Failed']

comparison['Difference (%)'] = (
    (comparison['Failed'] - comparison['Normal'])
    / comparison['Normal']
) * 100

print(comparison.round(3))

#We want to quantify how much each sensor variable changes between normal and failed machines.

In [ ]:
pd.crosstab(
    df['Type'],
    df['Machine failure'],
    normalize='index'
) * 100

In [ ]:
sns.countplot(
    data=df,
    x='Type',
    hue='Machine failure'
)

plt.title('Machine Failure by Product Type')
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=df,
    x='Rotational speed [rpm]',
    y='Torque [Nm]',
    hue='Machine failure',
    alpha=0.6
)

plt.title('Torque vs Rotational Speed by Machine Failure')
plt.show()

#This will help us visually investigate whether combinations of these two variables distinguish failed machines.

In [ ]:
failed_machines = df[df['Machine failure'] == 1]

plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=failed_machines,
    x='Rotational speed [rpm]',
    y='Torque [Nm]',
    color='green',
    alpha=0.7
)

plt.title('Torque vs Rotational Speed — Failed Machines Only')
plt.show()

## 5. Feature selection and leakage prevention

The target is `Machine failure`.

The five failure-mode columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) describe failure outcomes and are excluded from the input features to prevent target leakage. Identifier columns are also excluded.

In [ ]:
X = df.drop(columns=[
    'UDI',
    'Product ID',
    'Machine failure',
    'TWF',
    'HDF',
    'PWF',
    'OSF',
    'RNF'
])

y = df['Machine failure']

print(X.head())
print("Shape:", X.shape)

## 6. Train, validation, and test split

A stratified 70% / 15% / 15% split preserves the rare failure class in all three partitions.

- Training set: model fitting and cross-validation
- Validation set: decision-threshold selection
- Test set: final holdout evaluation only

In [ ]:
from sklearn.model_selection import train_test_split

# First split: 70% training, 30% remaining
X_train_new, X_remaining, y_train_new, y_remaining = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

# Second split: divide remaining data equally
X_val_new, X_test_new, y_val_new, y_test_new = train_test_split(
    X_remaining,
    y_remaining,
    test_size=0.50,
    stratify=y_remaining,
    random_state=42
)

print("Training set:", X_train_new.shape)
print("Validation set:", X_val_new.shape)
print("Test set:", X_test_new.shape)

print("\nFailure distribution:")

print("Training:")
print(y_train_new.value_counts(normalize=True))

print("\nValidation:")
print(y_val_new.value_counts(normalize=True))

print("\nTest:")
print(y_test_new.value_counts(normalize=True))

## 7. Preprocessing pipeline and Random Forest tuning

Preprocessing is placed inside an sklearn `Pipeline`, so scaling and one-hot encoding are fitted within each cross-validation training fold. Grid search optimizes Average Precision, which is useful for this imbalanced classification problem.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Identify numerical and categorical columns
numerical_features = X_train_new.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

categorical_features = X_train_new.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Numerical features:", numerical_features)
print("Categorical features:", categorical_features)

# Preprocessing
preprocessor_new = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features)
    ]
)

# Complete ML pipeline
rf_pipeline_new = Pipeline(
    steps=[
        ("preprocessor", preprocessor_new),
        ("classifier", RandomForestClassifier(random_state=42))
    ]
)

# Hyperparameter search
param_grid_new = {
    "classifier__n_estimators": [300, 500],
    "classifier__max_depth": [10, 20, None],
    "classifier__min_samples_split": [2, 5],
    "classifier__min_samples_leaf": [1, 2],
    "classifier__class_weight": [None, "balanced_subsample"]
}

cv_new = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

grid_search_new = GridSearchCV(
    estimator=rf_pipeline_new,
    param_grid=param_grid_new,
    scoring="average_precision",
    cv=cv_new,
    n_jobs=-1,
    verbose=1
)

# Train using training data ONLY
grid_search_new.fit(X_train_new, y_train_new)

best_rf_new = grid_search_new.best_estimator_

print("\nBest parameters:")
print(grid_search_new.best_params_)

print("\nBest cross-validation Average Precision:")
print(grid_search_new.best_score_)

## 8. Decision-threshold selection

The default classification threshold of 0.5 is not assumed. Instead, select the threshold on the validation set that maximizes precision while maintaining at least 80% recall.

In [ ]:
from sklearn.metrics import (
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score
)

# Get failure probabilities from the validation dataset
y_val_prob = best_rf_new.predict_proba(X_val_new)[:, 1]

# Calculate precision-recall values
precision, recall, thresholds = precision_recall_curve(
    y_val_new,
    y_val_prob
)

# Find thresholds that achieve at least 80% recall
valid_indices = np.where(recall[:-1] >= 0.80)[0]

if len(valid_indices) == 0:
    print("No threshold achieves 80% recall.")

else:
    # Select the threshold with the highest precision
    best_index = valid_indices[
        np.argmax(precision[valid_indices])
    ]

    best_threshold_new = thresholds[best_index]

    # Make predictions using selected threshold
    y_val_pred = (
        y_val_prob >= best_threshold_new
    ).astype(int)

    print("Selected threshold:", best_threshold_new)
    print("Validation Precision:",
          precision_score(y_val_new, y_val_pred))
    print("Validation Recall:",
          recall_score(y_val_new, y_val_pred))
    print("Validation F1:",
          f1_score(y_val_new, y_val_pred))

## 9. Final evaluation on the test set

This is the final evaluation of the selected model and validation-derived threshold. Do not use these test results to make further tuning decisions.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# 1. Predict probabilities on untouched test data
y_test_prob = best_rf_new.predict_proba(X_test_new)[:, 1]

# 2. Apply the threshold selected on validation data
y_test_pred = (
    y_test_prob >= best_threshold_new
).astype(int)

# 3. Calculate evaluation metrics
print("FINAL TEST RESULTS")
print("--------------------------")

print("Threshold:", best_threshold_new)
print("Accuracy:", accuracy_score(y_test_new, y_test_pred))
print("Precision:", precision_score(y_test_new, y_test_pred))
print("Recall:", recall_score(y_test_new, y_test_pred))
print("F1 Score:", f1_score(y_test_new, y_test_pred))
print("Average Precision:", average_precision_score(y_test_new, y_test_prob))
print("ROC-AUC:", roc_auc_score(y_test_new, y_test_prob))

# 4. Classification report
print("\nClassification Report:")
print(classification_report(y_test_new, y_test_pred))

# 5. Confusion matrix
cm = confusion_matrix(y_test_new, y_test_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Healthy", "Failure"]
)

disp.plot(cmap="Greens")
plt.title("Final Test Confusion Matrix")
plt.show()

## 10. Random Forest feature importance

In [ ]:
# Extract the fitted Random Forest
rf_classifier_new = best_rf_new.named_steps["classifier"]

# Get transformed feature names
feature_names_new = best_rf_new.named_steps[
    "preprocessor"
].get_feature_names_out()

# Extract feature importance
importance_df_new = pd.DataFrame({
    "Feature": feature_names_new,
    "Importance": rf_classifier_new.feature_importances_
})

importance_df_new = importance_df_new.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df_new)

# Visualize
importance_df_new.plot(
    x="Feature",
    y="Importance",
    kind="barh",
    figsize=(10, 6),
    legend=False
)

plt.title("Feature Importance - Corrected Random Forest")
plt.xlabel("Importance")
plt.gca().invert_yaxis()
plt.show()

## 11. SHAP explainability

SHAP values are calculated on the validation data to explain how transformed input features contribute to the model's failure-class output. These are model explanations, not proof of physical causation.

In [ ]:
import shap

# Extract fitted preprocessing and classifier
preprocessor_new = best_rf_new.named_steps["preprocessor"]
rf_classifier_new = best_rf_new.named_steps["classifier"]

# Transform validation data
X_val_transformed = preprocessor_new.transform(X_val_new)

# Convert sparse matrix if necessary
if hasattr(X_val_transformed, "toarray"):
    X_val_transformed = X_val_transformed.toarray()

# Get transformed feature names
feature_names_new = preprocessor_new.get_feature_names_out()

# Create SHAP explainer
explainer_new = shap.TreeExplainer(rf_classifier_new)

# Calculate SHAP values
shap_values_new = explainer_new.shap_values(
    X_val_transformed
)

# Handle class 1 (machine failure)
if isinstance(shap_values_new, list):
    failure_shap_new = shap_values_new[1]
else:
    shap_array_new = np.asarray(shap_values_new)

    if shap_array_new.ndim == 3:
        failure_shap_new = shap_array_new[:, :, 1]
    else:
        failure_shap_new = shap_array_new

print("SHAP shape:", failure_shap_new.shape)
print("Feature names:", feature_names_new)

In [ ]:
shap.summary_plot(
    failure_shap_new,
    X_val_transformed,
    feature_names=feature_names_new
)

## 12. Save the model and selected threshold

Save the fitted preprocessing-plus-classifier pipeline and the threshold selected on validation data. The API loads these two artifacts for inference.

In [ ]:
import joblib
import json

MODEL_DIR = BASE_DIR / "models"
MODEL_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODEL_DIR / "predictive_maintenance_model.joblib"
THRESHOLD_PATH = MODEL_DIR / "threshold.json"

joblib.dump(best_rf_new, MODEL_PATH)

with open(THRESHOLD_PATH, "w") as file:
    json.dump(
        {"threshold": float(best_threshold_new)},
        file,
        indent=4
    )

# Verify the saved model can be loaded
loaded_model = joblib.load(MODEL_PATH)

print("Model saved:", MODEL_PATH)
print("Threshold saved:", THRESHOLD_PATH)
print("Selected threshold:", float(best_threshold_new))
print("Model reload verification: successful")